The goal is to get an overall sense of how privacy-sensitive a database is and which attributes make it isensitive.  

Follows directly the method in :
S.-H. Kim, C. Jung, and Y.-J. Lee, “An entropy-based analytic model for the privacy-preserving in open data,” in 2016 IEEE International Conference on Big Data (Big Data), Dec. 2016, pp. 3676–3684. doi: 10.1109/BigData.2016.7841035.


Find out how likely it is that the attacker can match all the quasi identifiers to get the sensitive attribute:

i_entity: is the information that attacker has, bascially everything except the sensitive attribute

l_entity: the database we want to release with the sensitive data.  


In [22]:
import math

## 1. Load Dataset

In [ ]:
import pandas as pd

df = pd.read_csv("synthetic_dataset.csv")
print(f"Loaded dataset: {df.shape[0]} rows × {df.shape[1]} columns")
df.head()

Loaded dataset: 1000 rows × 6 columns


,text_message,url,language,country,device,star_review
0,"Exactly as described, fast delivery.",https://www.mystore.co/review/789,Spanish,Germany,Mobile,4
1,"Great product, really happy with it!",https://www.quickbuy.biz/item/456,Spanish,China,Desktop,5
2,Decent value for the price.,https://www.quickbuy.biz/listing/654,German,Brazil,Mobile,5
3,The item looks different from the photo.,https://www.quickbuy.biz/review/789,English,France,Mobile,4
4,The item looks different from the photo.,https://www.buynow.shop/listing/654,English,United Kingdom,Tablet,3


## 2. Column Classification

In [16]:
all_columns = list(df.columns)
print("Columns:", all_columns)

column_classification = {
    "semi_identifiers": ['url','language', 'country', 'device', 'star_review'],  #  fill in
    "sensitive":        ['text_message'],  # fill in
}

print("Semi-identifiers:", column_classification["semi_identifiers"])
print("Sensitive:       ", column_classification["sensitive"])

Columns: ['text_message', 'url', 'language', 'country', 'device', 'star_review']
Semi-identifiers: ['url', 'language', 'country', 'device', 'star_review']
Sensitive:        ['text_message']


I-Entity Dataset

The I-entity dataset is formed by dropping the sensitive attribute(s), retaining only identifiers and semi-identifiers.

In [ ]:
# TODO  For a actual test, use a different dataset, but this is what the paper does.
df_l_entity = df # The database we will release with the sensitive data
df_i_entity = df[column_classification["semi_identifiers"]].copy()  # what the attacker knows, with only the semi_identifiers

print(f"Original dataset : {df.shape}")
print(f"I-entity dataset : {df_i_entity.shape}")
print(f"Dropped columns  : {column_classification['sensitive']}")
df_i_entity.head()

Original dataset : (1000, 6)
I-entity dataset : (1000, 5)
Dropped columns  : ['text_message']


,url,language,country,device,star_review
0,https://www.mystore.co/review/789,Spanish,Germany,Mobile,4
1,https://www.quickbuy.biz/item/456,Spanish,China,Desktop,5
2,https://www.quickbuy.biz/listing/654,German,Brazil,Mobile,5
3,https://www.quickbuy.biz/review/789,English,France,Mobile,4
4,https://www.buynow.shop/listing/654,English,United Kingdom,Tablet,3


## 4. Matching Probabilities

Group the S-entity dataset by quasi-identifiers (semi-identifiers) to form **equivalence classes** — sets of records that are indistinguishable on those attributes.

For a target I-entity, the **matching probability** is the chance an attacker correctly links one record in its equivalence class to that individual:

$$P_{match} = \frac{1}{|equivalence\ class|}$$

A smaller class → higher matching probability → greater re-identification risk.

In [ ]:
# Data exploration only, this is also done again in the matching_probabilities function.

asi_ids = column_classification["semi_identifiers"]

# Size of each equivalence class
equiv_classes = (
    df_i_entity
    .groupby(quasi_ids)
    .size()
    .reset_index(name="class_size")
)
equiv_classes["match_probability"] = 1 / equiv_classes["class_size"]

equiv_classes = equiv_classes.sort_values("class_size").reset_index(drop=True)

print(f"Total equivalence classes : {len(equiv_classes)}")
print(f"Smallest class size       : {equiv_classes['class_size'].min()}")
print(f"Largest class size        : {equiv_classes['class_size'].max()}")
print(f"Mean class size           : {equiv_classes['class_size'].mean():.2f}")
equiv_classes.head(10)

Total equivalence classes : 956
Smallest class size       : 1
Largest class size        : 3
Mean class size           : 1.05


,url,language,country,device,star_review,class_size,match_probability
0,https://www.buynow.shop/item/456,Arabic,China,Desktop,5,1,1.0
1,https://www.rateit.app/listing/654,French,France,Mobile,2,1,1.0
2,https://www.rateit.app/listing/654,French,France,Mobile,4,1,1.0
3,https://www.rateit.app/listing/654,French,United States,Desktop,3,1,1.0
4,https://www.rateit.app/listing/654,German,India,Desktop,5,1,1.0
5,https://www.rateit.app/listing/654,Mandarin,Mexico,Desktop,2,1,1.0
6,https://www.rateit.app/listing/654,Mandarin,Mexico,Desktop,3,1,1.0
7,https://www.rateit.app/listing/654,Mandarin,Mexico,Mobile,3,1,1.0
8,https://www.rateit.app/listing/654,Mandarin,Mexico,Mobile,4,1,1.0
9,https://www.rateit.app/listing/654,Mandarin,United States,Mobile,1,1,1.0


In [19]:
def matching_probabilities(
    s_df: pd.DataFrame,
    i_df: pd.DataFrame,
    quasi_ids: list[str],
    sensitive_col: str,
) -> pd.DataFrame:
    """Calculate P(S_i, I_j) for every I-entity / sensitive-value pair.

    For each record I_j in i_df, finds the matching equivalence class in s_df
    (records sharing the same quasi-identifier values) and computes the
    probability of each sensitive attribute value within that class.

    Args:
        s_df:          S-entity dataset — contains quasi-identifiers and the
                       sensitive attribute column: this is what we want to share
        i_df:          I-entity dataset — contains only quasi-identifier columns: this is the rest of the world's information
        quasi_ids:     Column names to group on (quasi-identifiers / semi-identifiers).
        sensitive_col: Name of the sensitive attribute column in s_df.

    Returns:
        DataFrame with columns:
            i_entity_index   — original row index from i_df
            <quasi_id cols>  — the quasi-identifier values for that I-entity
            sensitive_value  — a distinct value of the sensitive attribute
            match_count      — how many S-entity records in the group have this value
            group_size       — total records in the matched equivalence class
            probability      — match_count / group_size  (i.e. P(S_i, I_j))
        Unmatched I-entities (no group in s_df) are included with NaN probability.
    """
    # Pre-compute frequency distribution for every equivalence class in s_df
    group_counts = (
        s_df.groupby(quasi_ids + [sensitive_col])
        .size()
        .reset_index(name="match_count")
    )
    equiv_classes = (
        s_df.groupby(quasi_ids)
        .size()
        .reset_index(name="class_size")
    )
    group_dist = group_counts.merge(equiv_classes, on=quasi_ids)
    group_dist["probability"] = group_dist["match_count"] / group_dist["class_size"]

    # Tag each I-entity with its original index, then merge on quasi-identifiers
    i_tagged = i_df[quasi_ids].copy()
    i_tagged.insert(0, "i_entity_index", i_tagged.index)

    result = i_tagged.merge(group_dist, on=quasi_ids, how="left")
    #result = result.rename(columns={sensitive_col: "sensitive_value"})

    return result.reset_index(drop=True)


# --- Example call (fill in column names once classification is complete) ---
sensitive_col = column_classification["sensitive"][0]


prob_df = matching_probabilities(df_l_entity, df_i_entity, column_classification["semi_identifiers"], sensitive_col)
print(f"Result shape: {prob_df.shape}")
print(prob_df.head(20).to_string(index=False))


Result shape: (1086, 10)
 i_entity_index                                   url   language        country  device  star_review                             text_message  match_count  class_size  probability
              0     https://www.mystore.co/review/789    Spanish        Germany  Mobile            4     Exactly as described, fast delivery.            1           1          1.0
              1     https://www.quickbuy.biz/item/456    Spanish          China Desktop            5     Great product, really happy with it!            1           1          1.0
              2  https://www.quickbuy.biz/listing/654     German         Brazil  Mobile            5              Decent value for the price.            1           1          1.0
              3   https://www.quickbuy.biz/review/789    English         France  Mobile            4 The item looks different from the photo.            1           1          1.0
              4   https://www.buynow.shop/listing/654    English United Kin

## 5. Entropy Measures per I-Entity

Three measures are computed for each I-entity $I_j$ from its sensitive attribute probability distribution:

| Measure | Formula | Interpretation |
|---|---|---|
| **Matching entropy** $H(S\|I_j)$ | $-\sum_i P(S_i, I_j)\log_2 P(S_i, I_j)$ | Uncertainty of linking $I_j$ to a sensitive value |
| **Maximum entropy** $H_{max}$ | $\log_2(n)$ where $n$ = unique sensitive values | Upper bound — perfectly uniform distribution |
| **Relative entropy** $D$ | $H_{max} - H(S\|I_j)$ | How far $I_j$'s distribution is from maximum uncertainty; higher = more re-identification risk |

In [24]:
def entropy_measures(prob_df: pd.DataFrame, sensitive_value) -> pd.DataFrame:
    """Calculate H, H_max, and D for each I-entity from matching_probabilities output.

    Args:
        prob_df: Output of matching_probabilities() — one row per (I_j, S_i) pair
                 with columns i_entity_index, sensitive_value, probability.

    Returns:
        DataFrame with one row per I-entity and columns:
            i_entity_index  — original index from i_df
            H               — matching entropy H(S|I_j) in bits
            H_max           — log2(n_unique_sensitive_values)
            D               — relative entropy H_max - H
    """
    n_sensitive = prob_df[sensitive_col].nunique()
    h_max = math.log2(n_sensitive) if n_sensitive > 1 else 0.0

    def _h(group):
        p = group["probability"].dropna()
        # Only sum terms where p > 0 (log2(0) is undefined)
        return -(p[p > 0] * p[p > 0].apply(math.log2)).sum()

    matching_entropy = (
        prob_df.groupby("i_entity_index")
        .apply(_h, include_groups=False)
        .reset_index(name="H")
    )

    matching_entropy["H_max"] = h_max
    matching_entropy["D"] = matching_entropy["H_max"] - matching_entropy["H"]

    return matching_entropy


if "prob_df" :
    entropy_df = entropy_measures(prob_df, sensitive_col)

    print(f"H_max (global)  : {entropy_df['H_max'].iloc[0]:.4f} bits")
    print(f"Mean H          : {entropy_df['H'].mean():.4f} bits")
    print(f"Mean D          : {entropy_df['D'].mean():.4f} bits")
    print(f"\nHighest-risk I-entities (lowest H, highest D):")
    print(entropy_df.sort_values("D", ascending=False).head(10).to_string(index=False))
else:
    print("Run the matching_probabilities cell first to produce prob_df.")

H_max (global)  : 4.3219 bits
Mean H          : 0.0843 bits
Mean D          : 4.2377 bits

Highest-risk I-entities (lowest H, highest D):
 i_entity_index    H    H_max        D
              0 -0.0 4.321928 4.321928
            655 -0.0 4.321928 4.321928
            642 -0.0 4.321928 4.321928
            643 -0.0 4.321928 4.321928
            644 -0.0 4.321928 4.321928
            645 -0.0 4.321928 4.321928
            646 -0.0 4.321928 4.321928
            648 -0.0 4.321928 4.321928
            649 -0.0 4.321928 4.321928
            650 -0.0 4.321928 4.321928


## 6. Re-identification Risk

- **Entity risk** — how close each I-entity's distribution is to being fully determined: $Risk_j = (D_j / H_{max}) \times 100$
- **Dataset risk** — the maximum entity risk, representing the weakest point in the dataset

In [25]:
entropy_df["risk_pct"] = (entropy_df["D"] / entropy_df["H_max"]) * 100

dataset_risk = entropy_df["risk_pct"].max()
worst = entropy_df.loc[entropy_df["risk_pct"].idxmax()]

print(f"Dataset re-identification risk : {dataset_risk:.2f}%")
print(f"Most vulnerable I-entity index : {int(worst['i_entity_index'])}")
print(f"  H      = {worst['H']:.4f} bits")
print(f"  H_max  = {worst['H_max']:.4f} bits")
print(f"  D      = {worst['D']:.4f} bits")
print(f"  Risk   = {worst['risk_pct']:.2f}%")

print("\nEntity risk distribution:")
print(entropy_df["risk_pct"].describe().round(2))

Dataset re-identification risk : 100.00%
Most vulnerable I-entity index : 0
  H      = -0.0000 bits
  H_max  = 4.3219 bits
  D      = 4.3219 bits
  Risk   = 100.00%

Entity risk distribution:
count    1000.00
mean       98.05
std         6.53
min        63.33
25%       100.00
50%       100.00
75%       100.00
max       100.00
Name: risk_pct, dtype: float64


## 7. Quasi-Identifier Sensitivity Analysis

Remove one quasi-identifier at a time and recompute entropy measures. This shows which column contributes most to re-identification risk — removing it causes the largest drop in dataset risk.

In [28]:
baseline_risk = dataset_risk  # already computed in section 6

sensitivity_records = []

for col in quasi_ids:
    reduced_quasi_ids = [q for q in quasi_ids if q != col]

    reduced_prob_df = matching_probabilities(df, df_i_entity, reduced_quasi_ids, sensitive_col)
    reduced_entropy_df = entropy_measures(reduced_prob_df, sensitive_col)
    reduced_entropy_df["risk_pct"] = (reduced_entropy_df["D"] / reduced_entropy_df["H_max"]) * 100

    sensitivity_records.append({
        "removed_column":  col,
        "mean_H":          reduced_entropy_df["H"].mean(),
        "mean_D":          reduced_entropy_df["D"].mean(),
        "dataset_risk_pct": reduced_entropy_df["risk_pct"].max(),
        "risk_delta":      reduced_entropy_df["risk_pct"].max() - baseline_risk,
    })

sensitivity_df = (
    pd.DataFrame(sensitivity_records)
    .sort_values("risk_delta")          # most negative = biggest risk reduction
    .reset_index(drop=True)
)

print(f"Baseline dataset risk : {baseline_risk:.2f}%\n")
print(sensitivity_df.round(4).to_string(index=False))

Baseline dataset risk : 100.00%

removed_column  mean_H  mean_D  dataset_risk_pct  risk_delta
           url  1.4007  2.9212             100.0         0.0
      language  0.3137  4.0082             100.0         0.0
       country  0.5220  3.7999             100.0         0.0
        device  0.1784  4.1435             100.0         0.0
   star_review  0.2760  4.0460             100.0         0.0


The attack is:  You know there is only one person who would search for roblox in a given location.  You find them in the database based on the quasi-identifiers and sensitive value.
Now, can you match the quasi-identifiers to get other sensitive values. 


In [ ]:
The attack is: you use a the search string similarity to create a session, so you know the 